<a href="https://colab.research.google.com/github/GSK7625/DO-AN-THI-GIAC-MAY-TINH/blob/kien/notebooks/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation for Crowd Counting

Notebook này hỗ trợ tải bộ dữ liệu ShanghaiTech từ Kaggle, giải nén và chuẩn bị dữ liệu cho huấn luyện YOLOv8/CSRNet.

## 1. Tải dữ liệu từ Kaggle

Để tải dữ liệu, bạn cần có file `kaggle.json` (API Token):
1. Vào [Kaggle Settings](https://www.kaggle.com/settings) -> **Create New API Token**.
2. Upload file `kaggle.json` vào Colab (nhấn vào biểu tượng thư mục ở bên trái rồi chọn Upload).

In [ ]:
# Cài đặt thư viện kaggle
!pip install -q kaggle

# Cấu hình thư mục chứa kaggle.json
import os
if not os.path.exists('/root/.kaggle'):
    !mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Tải và giải nén bộ dữ liệu
DATASET_ID = "tthien/shanghaitech"
TARGET_DIR = "/content/DO-AN-THI-GIAC-MAY-TINH/data/ShanghaiTech"

os.makedirs(TARGET_DIR, exist_ok=True)

print(f"Đang tải dataset {DATASET_ID}...")
!kaggle datasets download -d {DATASET_ID} -p {TARGET_DIR} --unzip

print("Tải và giải nén hoàn tất!")

## 2. Tiền xử lý dữ liệu

In [ ]:
import os
import glob
import scipy.io as sio
import cv2
from tqdm import tqdm
import yaml

In [ ]:
def convert_mat_to_yolo(part_path, box_size=15):
    image_dir = os.path.join(part_path, "images")
    gt_dir = os.path.join(part_path, "ground-truth")
    labels_dir = os.path.join(part_path, "labels")

    os.makedirs(labels_dir, exist_ok=True)

    img_paths = glob.glob(os.path.join(image_dir, "*.jpg"))
    print(f"Starting conversion for {len(img_paths)} images in {part_path}...")

    for img_path in tqdm(img_paths):
        img = cv2.imread(img_path)
        if img is None: continue
        h, w, _ = img.shape

        base_name = os.path.basename(img_path)
        mat_name = base_name.replace(".jpg", ".mat").replace("IMG_", "GT_IMG_")
        mat_path = os.path.join(gt_dir, mat_name)
        txt_path = os.path.join(labels_dir, base_name.replace(".jpg", ".txt"))

        if not os.path.exists(mat_path): continue

        mat = sio.loadmat(mat_path)
        points = mat["image_info"][0, 0][0, 0][0]

        with open(txt_path, "w") as f:
            for point in points:
                x, y = point[0], point[1]
                x_center, y_center = float(x) / w, float(y) / h
                box_w, box_h = float(box_size) / w, float(box_size) / h

                f.write(f"0 {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}\n")

In [ ]:
def create_yaml(data_root):
    yaml_content = {
        'path': os.path.abspath(data_root),
        'train': 'part_A/train_data/images',
        'val': 'part_A/test_data/images',
        'nc': 1,
        'names': ['head']
    }
    yaml_path = os.path.join(data_root, 'shanghaitech_partA.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, sort_keys=False)
    print(f"Created YOLO config at {yaml_path}")

In [ ]:
DATA_ROOT = "/content/DO-AN-THI-GIAC-MAY-TINH/data/ShanghaiTech"

for split in ["train", "test"]:
    path = os.path.join(DATA_ROOT, "part_A", f"{split}_data")
    if os.path.exists(path): convert_mat_to_yolo(path)

create_yaml(DATA_ROOT)